In [40]:
import os, json
import pandas as pd


# analyse the steering results
success schemas:
- counting 1s and -1s
    - counts of 1s of 2pos
    - counts of -1s of 2neg
- comparing the baseline
    - if baseline is 1, after 2neg, how many becomes -1 or 0?
    - if baseline is 0, 
        - after 2neg, how many becomes -1?
        - after 2pos, how many becomes 1?
    - if baseline is -1, after 2pos, how many becomes 0 or 1?
- counts of 1s for bridge

additionally for quality
- counts of 1s for repetition
- average fluency

In [41]:
# all files and dirs
baseline_path = "/scratch/fmeng/ActAdd/results/gemini_base/"
baseline_llama = "gemini_base_llama_senti+_fl_temp_0.json"
baseline_opt = "gemini_base_opt_senti+_fl_temp_0.json"
baseline_de = "gemini_base_de_senti+_fl.json"
baseline_zh = "gemini_base_zh_senti+_fl.json"

result_path = "/scratch/fmeng/ActAdd/results/"
dirs_llama = [
    "gemini_2pos_llama_senti+_fl_temp_0_no_space_hpt", 
    "gemini_2neg_llama_senti+_fl_temp_0_no_space_hpt", 
    "gemini_sent_2pos_llama_senti+_fl_temp_0_hpt", 
    "gemini_sent_2neg_llama_senti+_fl_temp_0_hpt"
    ]

dirs_opt = [
    "gemini_2pos_opt_senti+_fl_temp_0_no_space_hpt", 
    "gemini_2neg_opt_senti+_fl_temp_0_no_space_hpt", 
    "gemini_sent_2pos_opt_senti+_fl_temp_0_hpt",
    "gemini_sent_2neg_opt_senti+_fl_temp_0_hpt"
    ]

dirs_de = [
    "gemini_Love_de_senti+_fl",
    "gemini_Hate_de_senti+_fl",
    "gemini__love_de_senti+_fl",
    "gemini__hate_de_senti+_fl",
    "gemini_sent_2pos_de_senti+_fl",
    "gemini_sent_2neg_de_senti+_fl"
]

dirs_zh = [
    "gemini_Love_zh_senti+_fl",
    "gemini_Hate_zh_senti+_fl",
    "gemini__love_zh_senti+_fl",
    "gemini__hate_zh_senti+_fl",
    "gemini_sent_2pos_zh_senti+_fl",
    "gemini_sent_2neg_zh_senti+_fl"
]

dirs_bridge = [
    "gemini_bridge_llama_bridge+_fl_hpt", 
    "gemini_bridge_opt_bridge+_fl_hpt",
    "gemini_bridge_de_bridge+_fl",
    "gemini_bridge_zh_bridge+_fl"
    ]

# baseline
no generated sentence in the baseline is talking about the golden gate bridge

In [42]:
# baseline files have a different structure
def base_stats(dir, base_file):
    """
    return the sentiment labels of the base generation for downstream process
    """
    print("analysing ", base_file)
    df = pd.read_json(dir + base_file)
    print("counts of", df["continuation_label"].value_counts())
    print("number of repetitive sentences:", df["repetition"].sum().item())
    print("average perplexity of continuations:", df["fluency"].mean().item())
    print()
    return df["continuation_label"]

base_llama_sentimap = base_stats(baseline_path, baseline_llama)
base_opt_sentimap = base_stats(baseline_path, baseline_opt)
base_de_sentimap = base_stats(baseline_path, baseline_de)
base_zh_sentimap = base_stats(baseline_path, baseline_zh)

analysing  gemini_base_llama_senti+_fl_temp_0.json
counts of continuation_label
 0    13
 1     5
-1     2
Name: count, dtype: int64
number of repetitive sentences: 5
average perplexity of continuations: 2.557923251390457

analysing  gemini_base_opt_senti+_fl_temp_0.json
counts of continuation_label
 0    9
 1    7
-1    4
Name: count, dtype: int64
number of repetitive sentences: 12
average perplexity of continuations: 3.263213074207306

analysing  gemini_base_de_senti+_fl.json
counts of continuation_label
0    16
1     4
Name: count, dtype: int64
number of repetitive sentences: 11
average perplexity of continuations: 2.575465887784958

analysing  gemini_base_zh_senti+_fl.json
counts of continuation_label
 0    15
 1     3
-1     2
Name: count, dtype: int64
number of repetitive sentences: 7
average perplexity of continuations: 6.066083538532257



# sentiment
temperature 0
## harmonic mean

In [80]:
def dfs2hms(dfs):
    """
    harmonic mean
    """
    n = len(dfs)
    dfs_rec = list()
    for df in dfs:
        dfs_rec.append(1/df)
    sum_dfs_rec = sum(dfs_rec)
    hms = n/sum_dfs_rec
    return hms

def dfs2ams(dfs):
    """
    arithmetic mean
    """
    n = len(dfs)
    sum_dfs = sum(dfs)
    return sum_dfs/n

def get_means(success, repetition, fluency=None, mean=dfs2hms):
    """
    success: grid containing the counts of 1s or -1s, or increase relative to the baseline results
    repetition: 
    fluency: for opt results can be ignored
    https://www.datacamp.com/tutorial/how-to-normalize-data
    """
    # normalise success
    # https://pandas.pydata.org/docs/reference/api/pandas.DataFrame.clip.html
    success = (success/20).clip(0.00001, 1)
    # print("success", success)
    # normalise repetion
    diversity = ((20-repetition)/20).clip(0.00001, 1)
    # print("diversity", diversity)
    if fluency is not None: 
        fl_max = fluency.max().max()  # 315.0967122733593
        fl_min = fluency.min().min()  # 1.8930507719516754
        numberator = fl_max - fluency
        denominator = fl_max - fl_min
        fl_norm = (numberator/denominator).clip(0.00001, 1)
        # print("fluency", fl_norm)
        print("fluency counted")
        hms = mean([success, diversity, fl_norm])
    else:
        print("fluency ignored")
        hms = mean([success, diversity])
    hms = hms.apply(pd.to_numeric).astype(float)
    return hms

## counting 1s or -1s
counting the number of positive/negative

In [ ]:
# process files in the whole directory, counting 
def senti_stats(dir, include_fl=False):
    print("showing result for directory", dir)
    lst_file = os.listdir(f"{result_path}{dir}/")
    n_files = len(lst_file)
    grid_one = pd.DataFrame(0, index=range(n_files),columns=range(20))
    grid_zero = pd.DataFrame(0, index=range(n_files),columns=range(20))
    grid_neg = pd.DataFrame(0, index=range(n_files),columns=range(20))
    grid_rep = pd.DataFrame(0, index=range(n_files),columns=range(20))
    grid_fl = pd.DataFrame(index=range(n_files),columns=range(20))
    for file_name in lst_file:
        layer = int(file_name[file_name.rfind('_')+1:].split(".")[0])
        with open(f"{result_path}{dir}/{file_name}", "r") as f: 
            r_dict = json.load(f)
        for coeff in r_dict:
            list_dict = pd.DataFrame(r_dict[coeff])
            coeff = int(coeff)
            if 1 in list_dict["continuation_label"].value_counts():
                grid_one.loc[layer, coeff-1] = list_dict["continuation_label"].value_counts()[1]
            if 0 in list_dict["continuation_label"].value_counts():
                grid_zero.loc[layer, coeff-1] = list_dict["continuation_label"].value_counts()[0]
            if -1 in list_dict["continuation_label"].value_counts():
                grid_neg.loc[layer, coeff-1] = list_dict["continuation_label"].value_counts()[-1]
            grid_rep.loc[layer, coeff-1] = list_dict["repetition"].sum().item()
            grid_fl.loc[layer, coeff-1] = list_dict["fluency"].mean().item()
    # https://stackoverflow.com/questions/12286607/making-heatmap-from-pandas-dataframe
    # https://stackoverflow.com/questions/61363712/how-to-print-a-pandas-io-formats-style-styler-object
    print("count of repetitive sentences ↓")
    display(grid_rep.style.background_gradient(cmap='Reds', axis=None))
    print("average perplexity of continuations ↓")
    gmap_clipped = grid_fl.clip(upper=grid_fl.quantile(0.95), axis=1)
    display(
        grid_fl.style.background_gradient(cmap='Reds', gmap=gmap_clipped, axis=None).format("{:,.2f}")
    )
    if not include_fl:
        grid_fl = None
    if "_2pos" in dir or "__love" in dir or "_Love" in dir: 
        print("count of positive continuation ↑")
        display(grid_one.style.background_gradient(cmap='Blues', axis=None))
        print("harmonic mean ↑")
        hms = get_means(grid_one, grid_rep, grid_fl)
        display(hms.style.background_gradient(cmap='Blues', axis=None))
    if "_2neg" in dir or "_Hate" in dir or "__hate" in dir:
        print("count of negative continuation ↑")
        display(grid_neg.style.background_gradient(cmap='Blues', axis=None))
        print("harmonic mean ↑")
        hms = get_means(grid_neg, grid_rep, grid_fl)
        # hms = hms.apply(pd.to_numeric).astype(float)
        display(hms.style.background_gradient(cmap='Blues', axis=None))


In [49]:
for dir in dirs_zh:
    senti_stats(dir)

showing result for directory gemini_Love_zh_senti+_fl
count of repetitive sentences ↓


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,8,12,12,12,15,16,13,17,14,16,19,18,19,18,17,18,18,19,18,19
1,10,14,17,20,19,19,19,19,20,20,19,20,20,20,19,19,17,17,17,17
2,9,14,13,12,17,15,17,19,18,20,20,19,19,19,19,19,19,19,19,19
3,8,14,16,19,12,11,16,15,15,16,16,20,18,19,20,20,19,20,20,17
4,9,9,12,16,15,16,18,19,18,18,19,18,19,18,18,18,18,18,19,19
5,7,14,9,11,14,13,14,11,15,17,17,16,15,19,17,19,18,19,19,18
6,8,11,12,11,12,11,14,13,13,16,15,16,15,17,17,17,18,18,18,19
7,7,9,14,14,14,16,15,16,17,19,19,19,20,20,18,20,20,20,20,18
8,9,9,9,11,11,15,14,12,10,13,13,14,16,15,15,14,17,18,18,18
9,9,8,9,11,14,17,18,14,11,16,17,14,15,17,15,15,16,16,18,19


average perplexity of continuations ↓


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,5.55,5.58,5.21,4.69,4.50,5.75,5.26,5.64,6.60,6.07,5.73,5.76,4.70,4.60,5.57,6.79,6.66,5.38,5.33,5.09
1,5.71,4.10,5.21,4.70,18.01,3.67,3.27,3.50,3.14,3.99,204.95,2.91,3.10,3.45,272.24,137.66,391.21,401.69,"1,018.37",871.12
2,5.23,5.12,5.37,6.09,10.93,13.16,20.63,4.53,6.91,5.01,3.54,10.15,10.80,12.64,48.64,48.53,48.82,"2,435.34","2,435.71","2,435.75"
3,5.92,5.53,5.06,4.09,5.96,13.85,7.26,9.18,32.37,9.97,12.88,13.51,12.49,9.03,8.51,4.54,5.20,3.06,3.27,2.46
4,5.64,5.29,4.91,4.57,4.44,22.51,21.78,17.12,2.80,2.53,1.99,1.93,2.01,1.84,1.82,1.97,1.92,1.90,1.87,1.83
5,5.96,5.23,6.21,5.48,5.13,4.77,5.60,5.08,5.11,4.46,4.64,5.62,6.27,4.29,5.09,4.35,5.32,3.89,4.52,3.87
6,5.49,5.39,5.09,4.93,5.84,6.06,5.30,5.76,7.39,6.30,5.72,11.73,13.35,7.73,8.86,6.85,6.15,7.58,11.30,8.36
7,5.61,5.39,5.70,5.11,5.80,5.42,5.91,4.83,5.87,5.54,4.90,6.55,5.89,5.89,6.44,5.73,5.84,5.53,4.84,6.89
8,5.78,5.10,5.27,5.05,5.63,5.46,5.51,6.42,5.42,5.72,8.21,8.43,8.34,9.08,9.08,8.18,7.43,6.86,6.64,9.04
9,5.86,5.71,6.15,5.63,5.97,5.39,5.53,6.10,8.21,9.80,10.97,9.60,7.90,8.35,7.81,7.80,7.22,7.29,6.91,6.82


count of positive continuation ↑


...

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,5.69,5.31,5.98,4.92,5.47,5.14,5.70,5.31,5.50,5.20,5.31,5.31,5.27,5.20,5.17,5.42,5.81,6.03,6.04,6.01
1,5.98,5.70,6.19,5.99,5.87,5.71,6.12,5.75,5.13,5.86,5.79,5.71,5.87,5.32,5.45,5.22,5.28,5.62,5.84,5.72
2,5.54,5.63,5.51,5.27,5.27,5.15,5.77,4.81,4.84,5.11,4.21,3.94,4.18,4.32,4.41,5.04,4.61,4.70,5.25,5.06
3,5.52,5.43,5.44,5.73,5.33,5.78,5.24,5.51,5.66,6.21,5.60,5.44,5.30,5.74,5.19,5.59,5.53,5.73,6.11,6.26
4,5.43,4.97,5.64,17.97,9.60,10.68,10.12,7.03,7.43,6.39,7.85,9.39,5.98,6.69,100.61,8.64,8.28,8.02,7.46,9.31
5,5.58,5.44,5.90,6.44,5.59,5.93,5.72,5.88,6.08,5.95,5.95,5.75,5.74,6.07,5.58,5.25,5.64,5.43,5.62,5.86
6,5.77,5.70,5.16,5.13,5.37,5.68,4.69,4.41,4.42,4.91,5.04,4.88,4.73,4.93,5.04,5.08,5.60,6.66,6.49,6.41
7,5.50,5.06,4.91,4.60,5.24,5.54,5.53,5.89,7.11,7.84,8.24,11.73,10.29,12.56,13.77,22.55,18.71,18.58,24.02,26.99
8,5.89,5.37,5.06,4.67,4.70,5.09,5.04,4.94,4.62,5.02,4.68,4.75,4.42,4.64,4.61,5.13,5.70,5.49,4.92,5.32
9,5.41,4.97,5.41,5.88,5.43,5.02,4.70,4.99,5.61,5.36,4.91,5.14,4.97,5.18,5.35,5.80,5.64,5.72,6.44,6.26


count of negative continuation ↑


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,0,1,1,0,0,0,1,1,0,1,1,2,2,2,1,1,1,1,0,0
1,2,0,0,0,2,1,1,2,1,1,1,1,0,0,0,0,0,1,1,1
2,0,0,0,0,0,0,1,2,1,1,0,0,0,1,0,1,1,1,1,0
3,0,1,3,2,0,0,3,3,3,2,2,2,2,1,1,1,3,3,3,2
4,0,0,1,1,1,1,1,1,1,1,2,1,1,1,3,2,4,2,0,1
5,0,1,3,0,0,0,0,2,1,2,2,1,1,0,0,0,1,0,0,0
6,1,1,0,2,2,0,0,1,1,1,1,1,2,1,0,0,1,0,0,0
7,1,2,0,1,1,0,1,0,2,0,2,2,0,2,1,1,2,1,1,0
8,0,2,1,1,0,0,1,0,1,0,0,1,1,2,4,2,2,3,4,2
9,0,2,0,1,1,1,1,1,3,3,2,3,2,2,3,0,1,2,3,3


harmonic mean ↑
fluency ignored


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,0.000020,0.090909,0.090909,0.000020,0.000020,0.000020,0.091667,0.091667,0.000020,0.091667,0.090909,0.163636,0.166667,0.166667,0.088889,0.090000,0.090000,0.090000,0.000020,0.000020
1,0.173333,0.000020,0.000020,0.000020,0.169231,0.092857,0.092308,0.171429,0.092308,0.088889,0.090000,0.090000,0.000020,0.000020,0.000020,0.000020,0.000020,0.085714,0.083333,0.085714
2,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.091667,0.160000,0.088889,0.085714,0.000020,0.000020,0.000020,0.085714,0.000020,0.085714,0.090000,0.090000,0.087500,0.000020
3,0.000020,0.090909,0.230769,0.163636,0.000020,0.000020,0.218182,0.225000,0.235714,0.160000,0.163636,0.163636,0.163636,0.090000,0.087500,0.087500,0.218182,0.200000,0.225000,0.166667
4,0.000020,0.000020,0.088889,0.090909,0.090909,0.083333,0.075000,0.083333,0.085714,0.088889,0.166667,0.090000,0.090000,0.090000,0.225000,0.150000,0.240000,0.160000,0.000020,0.088889
5,0.000020,0.092308,0.235714,0.000020,0.000020,0.000020,0.000020,0.175000,0.091667,0.169231,0.155556,0.087500,0.091667,0.000020,0.000020,0.000020,0.092308,0.000020,0.000020,0.000020
6,0.091667,0.090909,0.000020,0.166667,0.163636,0.000020,0.000020,0.090000,0.085714,0.090909,0.091667,0.093333,0.169231,0.092308,0.000020,0.000020,0.091667,0.000020,0.000020,0.000020
7,0.091667,0.169231,0.000020,0.088889,0.090000,0.000020,0.091667,0.000020,0.171429,0.000020,0.160000,0.120000,0.000020,0.120000,0.075000,0.050000,0.066667,0.000020,0.050000,0.000020
8,0.000020,0.166667,0.090909,0.090909,0.000020,0.000020,0.091667,0.000020,0.091667,0.000020,0.000020,0.090000,0.090909,0.160000,0.276923,0.160000,0.160000,0.200000,0.266667,0.160000
9,0.000020,0.160000,0.000020,0.090000,0.090000,0.088889,0.088889,0.087500,0.225000,0.210000,0.133333,0.150000,0.155556,0.155556,0.200000,0.000020,0.080000,0.142857,0.171429,0.187500


## comparing with baseline

In [ ]:
# base_llama_sentimap[10].item()  # 0, 1, -1
def comparative_stats(dir, sentimap, include_fl=False):
    """
    base_llama_sentimap or base_opt_sentimap 
    gemini_2pos_llama_senti+_fl_temp_0_no_space_hpt
    """
    print("showing result for directory", dir)    
    lst_files = os.listdir(f"{result_path}{dir}/")
    n_files = len(lst_files)
    grid_success = pd.DataFrame(0, index=range(n_files),columns=range(20))
    grid_rep = pd.DataFrame(0, index=range(n_files),columns=range(20))
    grid_fl = pd.DataFrame(index=range(n_files),columns=range(20))
    if "_2pos" in dir or "__love" in dir or "_Love" in dir:  # count the tags that are larger than the corresponding one in the base_map
        for file_name in lst_files:
            layer = int(file_name[file_name.rfind('_')+1:].split(".")[0])
            with open(f"{result_path}{dir}/{file_name}", "r") as f: 
                r_dict = json.load(f)
            for coeff in r_dict:
                list_dict = pd.DataFrame(r_dict[coeff])
                coeff = int(coeff)
                # pointwise compare with llama_sentimap
                successs = list_dict["continuation_label"] > sentimap
                grid_success.loc[layer, coeff-1] = successs.sum()
                grid_rep.loc[layer, coeff-1] = list_dict["repetition"].sum().item()
                grid_fl.loc[layer, coeff-1] = list_dict["fluency"].mean().item()
    if "_2neg" in dir or "_Hate" in dir or "__hate" in dir:  # count the tags that are smaller than the corresponding one in the base_map
        for file_name in lst_files:
            layer = int(file_name[file_name.rfind('_')+1:].split(".")[0])
            with open(f"{result_path}{dir}/{file_name}", "r") as f: 
                r_dict = json.load(f)
            for coeff in r_dict:
                list_dict = pd.DataFrame(r_dict[coeff])
                coeff = int(coeff)
                successs = list_dict["continuation_label"] < sentimap
                grid_success.loc[layer, coeff-1] = successs.sum()
                grid_rep.loc[layer, coeff-1] = list_dict["repetition"].sum().item()
                grid_fl.loc[layer, coeff-1] = list_dict["fluency"].mean().item()
    print("count of bridges ↑")
    display(grid_success.style.background_gradient(cmap='Blues', axis=None))
    if not include_fl:
        grid_fl = None
    print("harmonic mean ↑")
    hms = get_means(grid_success, grid_rep, grid_fl)
    # hms = hms.apply(pd.to_numeric).astype(float)
    display(hms.style.background_gradient(cmap='Blues', axis=None))


In [54]:
for dir in dirs_zh:
    comparative_stats(dir, base_zh_sentimap)

showing result for directory gemini_Love_zh_senti+_fl
count of bridges ↑


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,2,4,6,3,3,4,4,4,4,3,3,4,3,5,7,6,4,5,5,3
1,5,4,5,6,3,3,2,3,4,4,4,4,3,4,4,4,3,3,3,4
2,2,5,3,5,6,5,6,6,4,4,5,4,4,4,4,4,6,5,4,2
3,1,5,3,4,6,4,6,9,10,3,3,5,4,4,2,3,4,2,2,2
4,2,4,4,5,5,5,3,4,3,2,3,2,2,3,3,3,3,3,4,4
5,4,6,9,3,3,3,4,4,6,5,6,3,5,3,6,4,4,3,4,3
6,3,4,3,4,4,4,4,4,6,4,5,5,8,4,3,3,3,4,5,4
7,4,4,3,3,5,6,6,5,3,7,4,4,4,5,6,5,7,6,6,6
8,3,4,5,5,2,3,4,7,5,5,6,5,5,7,5,6,6,7,7,7
9,1,4,2,2,5,4,8,5,4,2,5,6,5,5,6,7,6,7,7,7


harmonic mean ↑
fluency ignored


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,0.171429,0.266667,0.342857,0.218182,0.187500,0.200000,0.254545,0.171429,0.240000,0.171429,0.075000,0.133333,0.075000,0.142857,0.210000,0.150000,0.133333,0.083333,0.142857,0.075000
1,0.333333,0.240000,0.187500,0.000020,0.075000,0.075000,0.066667,0.075000,0.000020,0.000020,0.080000,0.000020,0.000020,0.000020,0.080000,0.080000,0.150000,0.150000,0.150000,0.171429
2,0.169231,0.272727,0.210000,0.307692,0.200000,0.250000,0.200000,0.085714,0.133333,0.000020,0.000020,0.080000,0.080000,0.080000,0.080000,0.080000,0.085714,0.083333,0.080000,0.066667
3,0.092308,0.272727,0.171429,0.080000,0.342857,0.276923,0.240000,0.321429,0.333333,0.171429,0.171429,0.000020,0.133333,0.080000,0.000020,0.000020,0.080000,0.000020,0.000020,0.120000
4,0.169231,0.293333,0.266667,0.222222,0.250000,0.222222,0.120000,0.080000,0.120000,0.100000,0.075000,0.100000,0.066667,0.120000,0.120000,0.120000,0.120000,0.120000,0.080000,0.080000
5,0.305882,0.300000,0.495000,0.225000,0.200000,0.210000,0.240000,0.276923,0.272727,0.187500,0.200000,0.171429,0.250000,0.075000,0.200000,0.080000,0.133333,0.075000,0.080000,0.120000
6,0.240000,0.276923,0.218182,0.276923,0.266667,0.276923,0.240000,0.254545,0.323077,0.200000,0.250000,0.222222,0.307692,0.171429,0.150000,0.150000,0.120000,0.133333,0.142857,0.080000
7,0.305882,0.293333,0.200000,0.200000,0.272727,0.240000,0.272727,0.222222,0.150000,0.087500,0.080000,0.080000,0.000020,0.000020,0.150000,0.000020,0.000020,0.000020,0.000020,0.150000
8,0.235714,0.293333,0.343750,0.321429,0.163636,0.187500,0.240000,0.373333,0.333333,0.291667,0.323077,0.272727,0.222222,0.291667,0.250000,0.300000,0.200000,0.155556,0.155556,0.155556
9,0.091667,0.300000,0.169231,0.163636,0.272727,0.171429,0.160000,0.272727,0.276923,0.133333,0.187500,0.300000,0.250000,0.187500,0.272727,0.291667,0.240000,0.254545,0.155556,0.087500


showing result for directory gemini_Hate_zh_senti+_fl
count of bridges ↑


...

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,0.218182,0.293333,0.225000,0.291667,0.266667,0.240000,0.333333,0.293333,0.240000,0.169231,0.171429,0.093333,0.173333,0.173333,0.311111,0.315789,0.315789,0.315789,0.311111,0.240000
1,0.166667,0.240000,0.307692,0.285714,0.240000,0.235714,0.230769,0.225000,0.266667,0.276923,0.266667,0.230769,0.276923,0.293333,0.285714,0.150000,0.150000,0.166667,0.171429,0.225000
2,0.173333,0.169231,0.171429,0.090000,0.276923,0.276923,0.300000,0.222222,0.266667,0.150000,0.080000,0.080000,0.000020,0.066667,0.120000,0.075000,0.075000,0.066667,0.000020,0.120000
3,0.166667,0.235714,0.285714,0.285714,0.150000,0.160000,0.218182,0.400000,0.300000,0.225000,0.166667,0.225000,0.225000,0.285714,0.225000,0.266667,0.333333,0.163636,0.230769,0.343750
4,0.090000,0.166667,0.291667,0.333333,0.343750,0.293333,0.333333,0.375000,0.444444,0.400000,0.393750,0.360000,0.343750,0.333333,0.293333,0.333333,0.240000,0.276923,0.240000,0.150000
5,0.230769,0.307692,0.200000,0.300000,0.200000,0.323077,0.240000,0.222222,0.150000,0.133333,0.240000,0.171429,0.171429,0.171429,0.171429,0.133333,0.100000,0.120000,0.133333,0.133333
6,0.235714,0.321429,0.276923,0.235714,0.225000,0.276923,0.276923,0.323077,0.423529,0.444444,0.388235,0.373333,0.291667,0.250000,0.222222,0.250000,0.272727,0.240000,0.200000,0.240000
7,0.092308,0.169231,0.155556,0.293333,0.266667,0.342857,0.291667,0.307692,0.160000,0.160000,0.411765,0.291667,0.323077,0.307692,0.272727,0.291667,0.222222,0.200000,0.200000,0.187500
8,0.173333,0.243750,0.169231,0.225000,0.218182,0.266667,0.171429,0.254545,0.350000,0.342857,0.323077,0.427778,0.300000,0.342857,0.373333,0.342857,0.350000,0.400000,0.342857,0.352941
9,0.091667,0.247059,0.171429,0.323077,0.343750,0.300000,0.276923,0.285714,0.393750,0.218182,0.254545,0.218182,0.307692,0.321429,0.240000,0.427778,0.442105,0.495238,0.495000,0.463158


showing result for directory gemini_sent_2neg_zh_senti+_fl
count of bridges ↑


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,0,1,1,0,0,0,1,2,1,2,2,3,3,3,2,3,3,3,1,1
1,1,0,1,1,2,2,2,2,1,0,0,0,0,0,0,0,0,0,0,0
2,0,1,0,0,1,0,1,2,0,0,2,1,2,2,1,2,2,2,1,0
3,0,1,3,2,1,0,2,2,2,2,2,2,2,0,1,1,2,3,3,2
4,0,0,0,1,0,0,0,0,0,1,1,0,0,0,1,1,4,2,1,1
5,0,0,3,0,1,1,1,2,1,2,2,2,1,1,0,1,1,0,0,0
6,1,1,1,3,3,2,1,2,2,0,1,2,3,2,2,2,2,0,0,0
7,2,2,2,2,3,2,3,2,3,2,4,4,1,3,3,3,3,3,3,2
8,1,2,2,2,2,2,2,2,3,3,1,1,1,3,3,2,1,3,3,2
9,0,0,1,2,2,2,2,1,2,2,3,2,2,2,4,2,3,2,3,3


harmonic mean ↑
fluency ignored


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,0.000020,0.090909,0.090909,0.000020,0.000020,0.000020,0.091667,0.169231,0.092308,0.169231,0.166667,0.225000,0.230769,0.230769,0.160000,0.225000,0.225000,0.225000,0.090000,0.090000
1,0.092857,0.000020,0.093750,0.092857,0.169231,0.173333,0.171429,0.171429,0.092308,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020
2,0.000020,0.088889,0.000020,0.000020,0.091667,0.000020,0.091667,0.160000,0.000020,0.000020,0.150000,0.085714,0.155556,0.150000,0.080000,0.150000,0.163636,0.163636,0.087500,0.000020
3,0.000020,0.090909,0.230769,0.163636,0.090000,0.000020,0.160000,0.163636,0.169231,0.160000,0.163636,0.163636,0.163636,0.000020,0.087500,0.087500,0.160000,0.200000,0.225000,0.166667
4,0.000020,0.000020,0.000020,0.090909,0.000020,0.000020,0.000020,0.000020,0.000020,0.088889,0.090909,0.000020,0.000020,0.000020,0.090000,0.085714,0.240000,0.160000,0.087500,0.088889
5,0.000020,0.000020,0.235714,0.000020,0.090909,0.092308,0.092308,0.175000,0.091667,0.169231,0.155556,0.155556,0.091667,0.091667,0.000020,0.092308,0.092308,0.000020,0.000020,0.000020
6,0.091667,0.090909,0.091667,0.230769,0.225000,0.169231,0.088889,0.163636,0.150000,0.000020,0.091667,0.175000,0.235714,0.171429,0.171429,0.163636,0.169231,0.000020,0.000020,0.000020
7,0.169231,0.169231,0.142857,0.160000,0.225000,0.166667,0.235714,0.169231,0.240000,0.155556,0.266667,0.171429,0.087500,0.150000,0.150000,0.075000,0.075000,0.000020,0.075000,0.100000
8,0.092308,0.166667,0.166667,0.166667,0.171429,0.169231,0.169231,0.166667,0.235714,0.235714,0.091667,0.090000,0.090909,0.218182,0.225000,0.160000,0.088889,0.200000,0.218182,0.160000
9,0.000020,0.000020,0.090909,0.163636,0.163636,0.160000,0.160000,0.087500,0.163636,0.155556,0.171429,0.120000,0.155556,0.155556,0.240000,0.155556,0.171429,0.142857,0.171429,0.187500


# talking about the bridge

In [ ]:
def bridge_stats(dir, include_fl=False):
    """
    counting the number of steered sentences that talk about the golden gate bridge 
    """
    print("showing result for directory", dir)
    lst_files = os.listdir(f"{result_path}{dir}/")
    n_file = len(lst_files)
    grid_bridge = pd.DataFrame(0, index=range(n_file),columns=range(20))
    grid_rep = pd.DataFrame(0, index=range(n_file),columns=range(20))
    grid_fl = pd.DataFrame(index=range(n_file),columns=range(20))
    for file_name in lst_files:
        layer = int(file_name[file_name.rfind('_')+1:].split(".")[0])
        with open(f"{result_path}{dir}/{file_name}", "r") as f: 
            r_dict = json.load(f)
        for coeff in r_dict:
            list_dict = pd.DataFrame(r_dict[coeff])
            coeff = int(coeff)
            if 1 in list_dict["bridge"].value_counts():
                grid_bridge.loc[layer, coeff-1] = list_dict["bridge"].value_counts()[1]
            else:
                grid_bridge.loc[layer, coeff-1] = 0
            grid_rep.loc[layer, coeff-1] = list_dict["repetition"].sum().item()
            grid_fl.loc[layer, coeff-1] = list_dict["fluency"].mean().item()
    # https://stackoverflow.com/questions/12286607/making-heatmap-from-pandas-dataframe
    # https://stackoverflow.com/questions/61363712/how-to-print-a-pandas-io-formats-style-styler-object

    print("count of repetitive sentences ↓")
    display(grid_rep.style.background_gradient(cmap='Reds', axis=None))
    print("average perplexity of continuations ↓")
    gmap_clipped = grid_fl.clip(upper=grid_fl.quantile(0.95), axis=1)
    display(
        grid_fl.style.background_gradient(cmap='Reds', gmap=gmap_clipped, axis=None).format("{:,.2f}")
    )
    print("count of sentences talking about the bridge ↑")
    display(grid_bridge.style.background_gradient(cmap='Blues', axis=None))
    if not include_fl:
        grid_fl = None
    print("harmonic mean ↑")
    hms = get_means(grid_bridge, grid_rep, grid_fl)
    display(hms.style.background_gradient(cmap='Blues', axis=None))


In [56]:
for dir in dirs_bridge:
    bridge_stats(dir)

showing result for directory gemini_bridge_llama_bridge+_fl_hpt
count of repetitive sentences ↓


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,5,11,14,18,18,19,20,20,20,20,20,19,20,20,20,20,20,20,20,20
1,7,4,14,16,19,19,20,19,20,18,20,20,20,20,20,20,20,20,20,20
2,8,6,8,13,12,11,10,11,10,11,11,14,12,18,19,20,20,20,20,20
3,10,7,10,8,11,8,6,5,6,10,3,7,10,16,20,19,20,20,20,19
4,7,7,10,10,10,13,8,12,18,20,19,20,19,19,20,20,20,20,20,20
5,8,10,9,7,15,13,13,15,19,20,20,20,19,19,19,19,20,20,20,20
6,5,11,8,8,4,9,14,14,16,17,16,17,19,19,18,18,18,19,19,19
7,9,10,10,9,13,14,16,16,14,17,18,19,19,18,19,20,20,20,20,19
8,5,7,10,4,7,11,12,15,18,17,16,18,18,19,20,20,20,19,19,19
9,10,6,8,11,8,9,9,11,14,17,18,18,18,18,19,19,19,18,17,18


average perplexity of continuations ↓


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,2.88,5.55,8.75,8.04,9.28,13.83,14.68,15.72,11.76,12.00,10.84,8.53,11.51,9.49,26.63,76.70,32.44,10.12,60.70,12.53
1,2.84,2.93,3.04,3.10,2.78,2.54,2.66,2.97,2.83,2.83,2.61,2.42,2.47,2.50,2.46,2.43,2.41,2.24,2.22,2.20
2,2.72,2.92,3.85,3.18,2.91,3.42,3.56,4.47,6.76,8.44,8.82,7.70,6.94,4.30,3.47,3.80,4.25,4.63,3.06,3.71
3,2.78,3.40,3.02,3.63,3.33,3.93,4.41,4.98,4.85,4.76,6.30,5.58,5.40,4.54,4.23,3.45,3.50,3.95,3.54,3.75
4,2.70,3.06,3.12,3.20,3.30,3.56,3.26,2.89,2.67,2.52,2.61,2.81,2.79,2.80,2.76,2.84,2.86,2.61,2.75,2.65
5,2.83,2.81,3.00,2.93,3.09,2.80,2.89,2.76,3.25,3.22,3.50,3.60,3.57,3.85,3.72,3.97,4.22,3.84,3.30,3.46
6,2.82,2.84,2.92,3.24,3.33,3.12,2.99,3.74,3.47,3.48,3.59,3.51,3.11,3.52,3.18,2.95,3.05,3.17,2.90,3.03
7,2.75,2.62,2.57,2.72,2.63,2.73,3.03,3.13,3.19,2.98,3.02,3.04,3.01,2.93,3.02,2.93,3.05,3.18,3.02,3.01
8,2.75,2.89,2.79,2.85,2.84,3.17,2.96,2.98,3.16,3.56,3.58,3.69,3.99,4.00,3.64,3.86,4.19,3.67,3.69,3.63
9,2.48,3.32,2.92,3.01,3.07,3.13,3.51,3.46,2.96,3.14,3.42,3.56,3.72,3.38,3.46,3.45,3.73,3.79,3.80,4.45


count of sentences talking about the bridge ↑


...

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,5.39,5.18,6.42,9.66,"1,089.37",30.92,"4,578.84","1,860,835.88","15,646.35","38,598.51",875.09,"1,833.84",110.22,560.45,811.08,798.00,807.16,"8,631.08","263,048.87","330,623.44"
1,4.89,5.69,6.31,5.45,4.84,5.04,3.60,3.13,3.31,3.20,2.57,2.65,2.55,2.60,2.82,2.69,2.69,2.97,3.32,3.61
2,5.59,4.66,5.73,4.74,4.38,4.60,3.74,3.02,3.18,3.74,4.43,3.61,2.47,2.42,2.46,2.87,3.27,2.89,3.17,3.62
3,6.08,5.73,4.92,5.28,6.99,21.79,22.12,16.79,32.51,14.35,8.68,5.15,6.17,4.62,3.64,3.90,4.32,4.90,4.97,5.98
4,5.51,4.77,4.36,4.88,5.33,5.27,4.32,5.38,5.37,4.70,4.94,5.86,4.97,66.90,69.12,67.56,70.34,71.52,70.00,69.60
5,5.33,5.02,4.58,4.69,4.87,4.67,5.33,5.73,6.26,5.50,4.65,6.16,7.02,16.97,15.85,112.84,113.59,116.22,115.75,115.18
6,4.89,5.45,4.81,4.26,4.11,4.76,4.27,4.97,6.01,8.37,10.22,12.94,14.45,214.34,219.21,212.04,215.66,209.90,206.00,274.92
7,5.72,6.05,5.40,5.53,5.23,5.44,5.32,6.21,10.41,10.80,19.42,18.81,20.80,14.19,15.48,11.60,12.74,14.09,13.56,20.32
8,5.87,5.83,5.89,5.10,5.99,7.25,6.86,10.26,12.25,14.58,14.50,14.63,14.42,15.05,15.34,10.29,12.41,12.09,12.12,13.06
9,5.73,5.30,5.65,5.19,5.68,5.76,8.59,7.90,7.60,10.24,17.70,12.31,12.90,10.14,15.08,14.81,17.66,23.62,23.79,13.34


count of sentences talking about the bridge ↑


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,4,5,5,3,7,1,2,3,5,0,1,1,0,0,0,0,0,1,3,3
1,3,4,6,2,6,7,3,4,4,5,2,1,0,0,0,0,0,0,3,3
2,1,6,5,7,8,8,8,6,3,3,6,4,3,5,7,9,10,12,12,12
3,1,4,8,8,6,7,4,6,4,5,10,16,13,12,17,14,12,14,12,11
4,0,5,7,5,1,0,1,3,6,9,7,6,4,5,5,7,5,6,6,7
5,0,3,6,2,0,3,4,8,10,10,13,12,9,8,10,8,7,7,9,9
6,0,4,6,7,5,4,6,8,8,8,4,8,10,8,9,6,5,4,5,4
7,0,1,3,2,2,4,2,2,1,2,2,2,2,2,2,2,1,2,1,1
8,0,1,2,3,2,4,0,2,1,1,1,2,3,4,2,3,2,2,2,2
9,0,1,0,1,1,0,0,2,1,3,3,2,3,3,2,2,2,4,5,7


harmonic mean ↑
fluency ignored


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,0.266667,0.307692,0.291667,0.187500,0.000020,0.000020,0.066667,0.075000,0.000020,0.000020,0.000020,0.050000,0.000020,0.000020,0.000010,0.000010,0.000020,0.066667,0.120000,0.150000
1,0.187500,0.300000,0.300000,0.120000,0.085714,0.087500,0.000020,0.000020,0.000020,0.000020,0.000020,0.050000,0.000010,0.000010,0.000010,0.000010,0.000010,0.000020,0.000020,0.000020
2,0.090909,0.388235,0.272727,0.427778,0.342857,0.342857,0.266667,0.200000,0.120000,0.075000,0.085714,0.080000,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020
3,0.090000,0.305882,0.480000,0.444444,0.342857,0.393750,0.222222,0.240000,0.200000,0.222222,0.333333,0.320000,0.361111,0.092308,0.094444,0.000020,0.000020,0.000020,0.000020,0.000020
4,0.000020,0.352941,0.427778,0.291667,0.083333,0.000020,0.080000,0.075000,0.240000,0.225000,0.155556,0.272727,0.080000,0.187500,0.187500,0.210000,0.222222,0.240000,0.272727,0.291667
5,0.000020,0.210000,0.342857,0.166667,0.000020,0.210000,0.222222,0.266667,0.333333,0.230769,0.173333,0.300000,0.276923,0.218182,0.166667,0.218182,0.254545,0.291667,0.276923,0.276923
6,0.000020,0.293333,0.375000,0.427778,0.291667,0.305882,0.388235,0.342857,0.307692,0.373333,0.276923,0.266667,0.230769,0.373333,0.393750,0.300000,0.272727,0.240000,0.272727,0.254545
7,0.000020,0.085714,0.225000,0.175000,0.150000,0.254545,0.166667,0.171429,0.091667,0.171429,0.171429,0.171429,0.173333,0.169231,0.171429,0.160000,0.090000,0.169231,0.091667,0.093333
8,0.000020,0.092857,0.166667,0.218182,0.166667,0.293333,0.000020,0.166667,0.091667,0.092308,0.093333,0.173333,0.240000,0.300000,0.166667,0.225000,0.169231,0.163636,0.160000,0.155556
9,0.000020,0.090000,0.000020,0.090000,0.092308,0.000020,0.000020,0.160000,0.087500,0.230769,0.225000,0.171429,0.243750,0.240000,0.171429,0.173333,0.173333,0.300000,0.352941,0.373333


# zh with deepseek-llm-7b-base

In [66]:
deepseek_path = "/scratch/fmeng/ActAdd/results/zh_deepseek/"
baseline_deepseek = "gemini_base_zh_fl_senti+.json"
dirs_deepseek = [
    "zh_deepseek/gemini_2pos_batch_zh_fl_senti+",
    "zh_deepseek/gemini_2neg_batch_zh_fl_senti+",
    "zh_deepseek/gemini_sent_2pos_batch_zh_fl_senti+",
    "zh_deepseek/gemini_sent_2neg_batch_zh_fl_senti+"
]
dirs_bridge_deepseek = "zh_deepseek/gemini_bridge_batch_zh_fl_bridge+"

In [67]:
base_deepseek_sentimap = base_stats(deepseek_path, baseline_deepseek)

analysing  gemini_base_zh_fl_senti+.json
counts of continuation_label
 0    14
 1     5
-1     1
Name: count, dtype: int64
number of repetitive sentences: 15
average perplexity of continuations: 3.8704586207866667



## counting 1s or -1s

In [68]:
for dir in dirs_deepseek:
    senti_stats(dir)

showing result for directory zh_deepseek/gemini_2pos_batch_zh_fl_senti+
count of repetitive sentences ↓


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,16,15,18,18,19,18,18,18,19,19,20,18,18,18,19,19,20,20,20,20
1,15,10,15,13,12,12,13,14,12,13,10,14,13,15,16,18,14,13,11,6
2,18,16,17,13,16,18,16,17,16,16,16,14,13,15,13,10,11,10,14,14
3,18,14,15,17,14,15,14,15,15,15,16,14,14,15,14,15,17,15,14,15
4,17,17,16,14,16,15,13,13,13,17,17,17,14,17,16,16,16,16,17,17
5,16,18,15,14,17,14,16,18,17,18,16,15,17,16,16,18,20,16,16,19
6,17,16,16,15,13,12,17,16,16,16,17,15,17,17,16,15,15,15,16,17
7,16,16,19,15,15,16,14,15,16,17,16,15,16,16,16,16,17,17,17,17
8,17,14,16,17,17,16,15,17,18,17,18,18,17,18,19,18,17,18,18,19
9,14,15,15,14,14,17,17,15,17,18,19,19,18,18,17,18,17,18,18,19


average perplexity of continuations ↓


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,4.01,7.39,4.73,5.37,29.28,13.10,13.88,12.67,"3,207.61","3,641.91","3,641.66",84.20,148.32,"1,208.44","1,271.52","1,271.54","1,187.30","1,205.97","1,187.78",88.13
1,5.31,5.22,4.48,4.49,4.64,4.51,4.43,4.97,5.47,5.47,5.59,5.59,5.90,16.72,31.88,30.86,"5,328.41",769.97,"4,037,386,983.35","4,047,371,835.99"
2,4.83,5.16,4.75,4.82,4.74,4.80,5.08,5.31,4.84,4.90,5.00,5.61,5.59,5.61,5.61,5.41,5.42,6.05,6.91,6.63
3,4.41,4.82,5.38,5.28,4.84,5.94,6.00,8.33,9.30,9.36,10.43,12.38,12.35,8.12,7.92,9.51,9.45,8.86,9.51,9.73
4,4.38,4.73,5.63,5.34,5.67,4.77,5.18,5.40,5.44,5.30,5.89,5.97,6.62,7.01,6.88,6.80,7.16,7.70,7.81,8.49
5,4.90,4.65,5.02,4.46,5.11,5.06,4.71,4.78,5.00,5.11,5.59,5.64,5.56,5.51,5.85,6.19,6.07,6.50,5.70,5.18
6,4.80,4.87,5.20,4.84,4.56,4.90,5.05,4.99,4.94,5.02,5.27,5.31,5.37,5.50,5.58,6.31,6.27,6.53,6.80,6.76
7,4.43,4.57,4.47,5.02,4.93,4.45,4.88,4.67,4.67,4.84,5.09,5.04,5.15,5.16,5.10,5.12,5.17,5.43,5.43,5.73
8,4.43,4.65,4.53,4.03,4.47,4.52,4.70,4.52,4.36,4.49,4.33,4.37,4.36,4.52,4.51,4.84,4.70,5.03,4.84,4.96
9,4.24,4.34,4.35,4.31,4.48,4.55,4.56,4.70,4.70,4.67,4.84,4.72,4.71,4.68,4.73,4.82,4.68,4.67,4.66,4.60


count of positive continuation ↑


...

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,3.83,3.45,3.61,3.50,3.64,3.63,3.61,3.82,3.95,4.02,4.09,4.23,4.13,4.48,4.76,4.99,5.56,5.35,5.43,4.82
1,3.82,3.62,3.75,4.17,3.92,5.19,5.57,6.97,5.15,4.42,4.30,3.81,4.81,4.58,4.52,7.46,6.17,6.26,7.15,5.61
2,3.80,3.61,3.77,3.90,4.09,4.21,4.12,3.67,3.71,4.27,4.16,4.08,4.37,4.13,3.95,4.10,4.02,4.75,4.70,5.91
3,3.86,3.79,3.75,3.43,3.88,3.85,3.93,3.57,4.20,4.23,4.09,4.03,4.20,4.30,6.12,6.56,6.31,6.66,5.89,8.41
4,3.83,3.97,3.82,4.19,4.21,3.84,4.53,4.89,4.97,5.13,4.74,5.28,4.93,4.86,4.87,4.94,5.37,5.06,5.01,4.71
5,3.83,3.92,4.28,4.50,4.21,4.29,4.42,4.62,4.55,5.59,6.93,9.05,5.84,8.48,7.91,8.01,12.17,13.54,10.29,17.95
6,4.22,3.73,3.74,3.71,3.93,4.09,4.39,4.38,4.84,4.73,4.56,5.69,5.01,5.43,7.96,4.71,4.81,4.87,5.39,5.09
7,3.74,4.00,3.93,4.60,6.68,5.60,5.89,8.65,7.76,12.21,10.80,13.89,8.91,9.05,10.43,10.55,11.41,14.53,12.60,12.19
8,3.96,4.10,4.46,4.98,4.57,4.55,3.98,4.21,4.31,4.48,4.47,4.02,4.30,4.03,4.16,4.50,4.47,4.67,6.45,5.81
9,3.95,4.16,3.97,3.88,4.43,4.13,4.34,4.57,4.52,4.92,4.92,4.78,4.72,4.96,4.73,5.13,5.02,5.26,6.17,5.25


count of negative continuation ↑


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
1,0,0,1,1,0,0,0,1,0,0,0,0,0,0,0,1,0,0,0,0
2,0,0,0,1,1,1,0,0,0,0,0,0,0,0,0,1,0,0,0,0
3,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
4,1,1,1,0,0,0,2,0,0,0,0,1,1,1,1,1,1,1,1,1
5,1,1,0,0,0,0,1,0,0,1,0,1,1,2,0,1,0,0,0,0
6,2,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0
7,1,1,1,0,3,0,1,1,1,1,0,1,0,0,0,0,0,1,0,0
8,0,0,0,0,0,0,0,1,1,1,0,0,0,0,0,0,0,0,0,0
9,1,0,0,0,1,2,2,1,1,0,1,0,0,0,0,1,2,2,2,1


harmonic mean ↑
fluency ignored


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,0.000020,0.000020,0.000020,0.075000,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020
1,0.000020,0.000020,0.075000,0.083333,0.000020,0.000020,0.000020,0.080000,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.080000,0.000020,0.000020,0.000020,0.000020
2,0.000020,0.000020,0.000020,0.066667,0.066667,0.075000,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.080000,0.000020,0.000020,0.000020,0.000020
3,0.000020,0.000020,0.083333,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020
4,0.080000,0.080000,0.085714,0.000020,0.000020,0.000020,0.142857,0.000020,0.000020,0.000020,0.000020,0.075000,0.083333,0.075000,0.075000,0.066667,0.075000,0.066667,0.075000,0.075000
5,0.080000,0.075000,0.000020,0.000020,0.000020,0.000020,0.066667,0.000020,0.000020,0.000020,0.000020,0.050000,0.050000,0.100000,0.000020,0.066667,0.000020,0.000020,0.000020,0.000020
6,0.133333,0.080000,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020
7,0.080000,0.066667,0.075000,0.000020,0.075000,0.000020,0.085714,0.083333,0.080000,0.083333,0.000020,0.075000,0.000020,0.000020,0.000020,0.000020,0.000020,0.050000,0.000020,0.000010
8,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.050000,0.050000,0.075000,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020
9,0.075000,0.000020,0.000020,0.000020,0.080000,0.133333,0.142857,0.088889,0.088889,0.000020,0.080000,0.000020,0.000020,0.000020,0.000020,0.080000,0.142857,0.150000,0.155556,0.085714


## comparing with baseline

In [69]:
for dir in dirs_deepseek:
    comparative_stats(dir, base_deepseek_sentimap)

showing result for directory zh_deepseek/gemini_2pos_batch_zh_fl_senti+
count of bridges ↑


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,5,4,4,5,3,1,2,1,1,2,3,5,4,3,3,2,3,3,3,3
1,3,3,5,7,5,5,5,5,3,4,3,3,3,3,2,4,4,3,3,4
2,1,7,4,8,3,2,2,3,3,3,5,4,3,1,4,4,5,5,8,6
3,3,4,4,3,6,6,4,5,4,7,6,7,7,7,3,5,6,7,6,6
4,4,3,5,3,4,4,4,3,4,4,5,4,6,4,3,4,6,7,7,7
5,5,4,4,4,7,7,6,5,5,7,8,7,6,6,7,6,6,6,4,4
6,2,1,2,2,2,3,4,5,8,6,6,5,6,6,6,6,4,4,5,4
7,4,2,1,2,3,1,2,4,2,2,2,2,3,3,3,4,4,4,4,4
8,2,3,2,2,4,4,2,2,4,4,5,4,4,4,4,4,5,5,4,4
9,3,3,1,2,2,2,2,3,2,2,2,2,3,3,3,2,2,3,3,3


harmonic mean ↑
fluency ignored


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,0.222222,0.222222,0.133333,0.142857,0.075000,0.066667,0.100000,0.066667,0.050000,0.066667,0.000020,0.142857,0.133333,0.120000,0.075000,0.066667,0.000020,0.000020,0.000020,0.000020
1,0.187500,0.230769,0.250000,0.350000,0.307692,0.307692,0.291667,0.272727,0.218182,0.254545,0.230769,0.200000,0.210000,0.187500,0.133333,0.133333,0.240000,0.210000,0.225000,0.311111
2,0.066667,0.254545,0.171429,0.373333,0.171429,0.100000,0.133333,0.150000,0.171429,0.171429,0.222222,0.240000,0.210000,0.083333,0.254545,0.285714,0.321429,0.333333,0.342857,0.300000
3,0.120000,0.240000,0.222222,0.150000,0.300000,0.272727,0.240000,0.250000,0.222222,0.291667,0.240000,0.323077,0.323077,0.291667,0.200000,0.250000,0.200000,0.291667,0.300000,0.272727
4,0.171429,0.150000,0.222222,0.200000,0.200000,0.222222,0.254545,0.210000,0.254545,0.171429,0.187500,0.171429,0.300000,0.171429,0.171429,0.200000,0.240000,0.254545,0.210000,0.210000
5,0.222222,0.133333,0.222222,0.240000,0.210000,0.323077,0.240000,0.142857,0.187500,0.155556,0.266667,0.291667,0.200000,0.240000,0.254545,0.150000,0.000020,0.240000,0.200000,0.080000
6,0.120000,0.080000,0.133333,0.142857,0.155556,0.218182,0.171429,0.222222,0.266667,0.240000,0.200000,0.250000,0.200000,0.200000,0.240000,0.272727,0.222222,0.222222,0.222222,0.171429
7,0.200000,0.133333,0.050000,0.142857,0.187500,0.080000,0.150000,0.222222,0.133333,0.120000,0.133333,0.142857,0.171429,0.171429,0.171429,0.200000,0.171429,0.171429,0.171429,0.171429
8,0.120000,0.200000,0.133333,0.120000,0.171429,0.200000,0.142857,0.120000,0.133333,0.171429,0.142857,0.133333,0.171429,0.133333,0.080000,0.133333,0.187500,0.142857,0.133333,0.080000
9,0.200000,0.187500,0.083333,0.150000,0.150000,0.120000,0.120000,0.187500,0.120000,0.100000,0.066667,0.066667,0.120000,0.120000,0.150000,0.100000,0.120000,0.120000,0.120000,0.075000


showing result for directory zh_deepseek/gemini_2neg_batch_zh_fl_senti+
count of bridges ↑


...

,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,0.133333,0.120000,0.200000,0.240000,0.250000,0.272727,0.250000,0.200000,0.187500,0.171429,0.150000,0.187500,0.133333,0.120000,0.150000,0.142857,0.142857,0.171429,0.150000,0.150000
1,0.150000,0.133333,0.142857,0.210000,0.187500,0.240000,0.240000,0.254545,0.291667,0.291667,0.323077,0.350000,0.254545,0.254545,0.187500,0.142857,0.272727,0.266667,0.373333,0.300000
2,0.120000,0.171429,0.133333,0.155556,0.200000,0.222222,0.200000,0.272727,0.250000,0.272727,0.240000,0.222222,0.187500,0.240000,0.187500,0.222222,0.272727,0.222222,0.272727,0.240000
3,0.120000,0.142857,0.142857,0.133333,0.187500,0.222222,0.250000,0.240000,0.210000,0.210000,0.266667,0.272727,0.240000,0.240000,0.187500,0.250000,0.250000,0.254545,0.254545,0.240000
4,0.080000,0.187500,0.187500,0.171429,0.250000,0.240000,0.240000,0.187500,0.150000,0.200000,0.150000,0.171429,0.150000,0.200000,0.155556,0.272727,0.250000,0.187500,0.187500,0.187500
5,0.080000,0.171429,0.066667,0.200000,0.120000,0.171429,0.272727,0.254545,0.254545,0.210000,0.272727,0.200000,0.250000,0.272727,0.250000,0.272727,0.291667,0.323077,0.222222,0.240000
6,0.075000,0.150000,0.250000,0.133333,0.200000,0.160000,0.160000,0.160000,0.200000,0.200000,0.222222,0.300000,0.240000,0.291667,0.254545,0.240000,0.272727,0.272727,0.307692,0.250000
7,0.133333,0.100000,0.171429,0.133333,0.200000,0.210000,0.150000,0.307692,0.266667,0.240000,0.150000,0.133333,0.085714,0.150000,0.080000,0.142857,0.150000,0.000020,0.000020,0.083333
8,0.133333,0.200000,0.150000,0.100000,0.222222,0.171429,0.187500,0.133333,0.200000,0.080000,0.075000,0.000020,0.000020,0.066667,0.120000,0.171429,0.222222,0.160000,0.155556,0.210000
9,0.133333,0.100000,0.080000,0.120000,0.240000,0.133333,0.171429,0.133333,0.222222,0.250000,0.272727,0.250000,0.222222,0.250000,0.222222,0.200000,0.240000,0.272727,0.272727,0.291667


showing result for directory zh_deepseek/gemini_sent_2neg_batch_zh_fl_senti+
count of bridges ↑


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,1,0,0,1,0,0,1,1,2,2,1,2,2,4,4,4,2,2,3,3
1,1,0,1,1,0,0,0,2,3,3,4,4,4,4,3,4,4,3,4,5
2,0,0,0,1,1,1,0,0,2,3,4,3,3,3,3,3,2,2,2,2
3,0,1,2,1,1,1,2,2,2,1,2,2,2,2,2,2,2,2,3,4
4,1,1,1,0,0,0,2,1,1,1,2,2,2,2,2,3,2,3,4,4
5,1,1,1,2,2,2,2,1,1,2,3,2,3,4,3,4,3,3,3,4
6,1,1,1,3,2,0,0,1,0,1,2,2,3,5,5,5,2,3,3,3
7,2,1,1,0,4,2,3,4,3,3,2,4,3,4,4,4,4,5,4,4
8,1,1,1,0,0,2,1,3,3,4,3,4,4,4,4,4,4,3,3,3
9,2,1,1,2,3,4,5,4,4,4,5,4,4,2,3,4,6,7,7,6


harmonic mean ↑
fluency ignored


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,0.083333,0.000020,0.000020,0.075000,0.000020,0.000020,0.083333,0.075000,0.142857,0.120000,0.075000,0.150000,0.150000,0.222222,0.240000,0.254545,0.142857,0.142857,0.150000,0.120000
1,0.075000,0.000020,0.075000,0.083333,0.000020,0.000020,0.000020,0.133333,0.150000,0.171429,0.171429,0.222222,0.200000,0.200000,0.200000,0.200000,0.200000,0.200000,0.200000,0.222222
2,0.000020,0.000020,0.000020,0.066667,0.066667,0.075000,0.000020,0.000020,0.133333,0.187500,0.200000,0.171429,0.120000,0.120000,0.200000,0.171429,0.133333,0.155556,0.133333,0.100000
3,0.000020,0.080000,0.142857,0.080000,0.087500,0.075000,0.120000,0.133333,0.155556,0.080000,0.142857,0.133333,0.133333,0.120000,0.142857,0.133333,0.100000,0.120000,0.075000,0.200000
4,0.080000,0.080000,0.085714,0.000020,0.000020,0.000020,0.142857,0.083333,0.083333,0.085714,0.142857,0.120000,0.142857,0.120000,0.120000,0.120000,0.120000,0.120000,0.171429,0.171429
5,0.080000,0.075000,0.066667,0.133333,0.066667,0.100000,0.100000,0.075000,0.066667,0.000020,0.075000,0.066667,0.075000,0.133333,0.120000,0.133333,0.150000,0.120000,0.120000,0.133333
6,0.080000,0.080000,0.050000,0.120000,0.100000,0.000020,0.000020,0.066667,0.000020,0.075000,0.133333,0.100000,0.171429,0.187500,0.250000,0.222222,0.133333,0.200000,0.171429,0.150000
7,0.133333,0.066667,0.075000,0.000020,0.080000,0.133333,0.200000,0.222222,0.171429,0.187500,0.100000,0.171429,0.120000,0.133333,0.080000,0.133333,0.133333,0.083333,0.080000,0.000020
8,0.075000,0.075000,0.075000,0.000020,0.000020,0.066667,0.066667,0.075000,0.075000,0.171429,0.171429,0.200000,0.171429,0.171429,0.133333,0.171429,0.171429,0.120000,0.120000,0.171429
9,0.120000,0.066667,0.075000,0.133333,0.171429,0.200000,0.250000,0.266667,0.266667,0.200000,0.222222,0.200000,0.200000,0.142857,0.171429,0.200000,0.272727,0.323077,0.350000,0.300000


## bridges

In [70]:
bridge_stats(dirs_bridge_deepseek)

showing result for directory zh_deepseek/gemini_bridge_batch_zh_fl_bridge+
count of repetitive sentences ↓


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,16,13,16,18,20,19,20,20,19,19,19,20,19,18,18,18,17,17,16,16
1,17,17,16,16,16,19,18,12,13,18,13,9,5,2,2,2,1,0,0,1
2,13,15,16,16,14,17,19,18,19,19,20,20,19,18,19,19,19,19,19,19
3,17,14,14,12,15,16,18,17,20,20,20,20,20,20,20,20,20,20,20,20
4,16,11,17,15,13,13,17,14,16,16,16,20,18,17,18,19,20,20,20,20
5,19,13,12,13,13,17,14,15,13,13,17,16,16,15,15,18,19,20,17,17
6,18,16,15,14,14,14,15,14,14,14,14,15,15,18,19,18,16,16,17,18
7,18,15,13,13,13,14,14,15,15,14,15,14,14,13,13,15,15,12,13,12
8,16,16,14,14,17,14,14,15,14,14,14,13,14,14,13,16,15,16,14,15
9,17,17,17,17,14,18,16,17,16,17,17,18,18,17,15,15,14,14,13,13


average perplexity of continuations ↓


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,4.00,5.45,4.45,6.47,16.03,6.44,6.53,6.03,8.40,13.93,6.58,5.96,8.46,5.16,104.65,104.50,5.56,4.99,5.97,5.38
1,4.62,6.28,"4,324.62","4,324.39","4,325.54",28.17,135.93,"1,747.59","2,047.94","4,483,975.41","4,062,228,420.04","4,076,104,826.77","4,166,328,421.82","4,435,005,039.05","4,310,775,000.10","4,449,537,517.71","4,449,545,746.55","4,452,441,170.46","4,453,442,921.56","4,450,114,423.43"
2,4.59,5.46,7.15,5.04,5.39,11.81,6.77,9.02,204.01,6.80,10.27,10.20,8.76,7.94,6.76,8.24,11.88,7.36,7.22,12.95
3,4.35,5.39,5.89,5.53,5.75,5.75,4.70,5.02,4.80,5.43,4.77,6.29,5.30,5.13,4.92,7.12,10.35,9.69,10.40,8.14
4,4.12,5.36,5.20,6.29,7.67,7.58,6.93,8.07,8.24,8.59,9.14,8.21,8.42,11.94,11.37,10.94,12.80,15.08,14.33,17.17
5,4.47,5.40,5.79,5.50,7.64,6.43,7.78,10.76,8.80,8.94,9.20,10.00,11.66,8.72,8.57,8.41,7.26,7.94,8.27,7.48
6,4.57,4.69,5.09,6.68,6.92,6.12,5.61,5.35,5.27,5.30,5.32,5.24,5.33,5.51,5.73,6.33,6.49,6.48,6.54,6.72
7,4.69,4.87,5.74,6.37,6.27,6.18,5.38,5.89,5.75,6.83,6.06,6.15,6.27,6.10,6.58,6.27,6.98,6.83,7.56,7.69
8,4.60,4.93,4.53,4.67,4.70,4.73,4.90,4.92,5.72,6.49,6.40,7.08,6.47,6.03,6.76,7.81,7.94,8.74,8.07,9.02
9,4.82,4.75,5.05,4.90,4.56,4.78,4.66,4.57,5.21,5.28,5.55,6.23,6.31,6.07,6.00,6.84,6.93,7.68,7.20,7.10


count of sentences talking about the bridge ↑


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,0,1,0,1,0,0,0,0,0,0,0,0,0,0,1,1,2,1,1,1
1,0,0,0,0,1,1,1,6,6,5,9,10,11,18,18,19,20,20,20,18
2,0,0,0,0,0,0,0,2,3,4,6,7,8,11,11,11,9,9,9,10
3,0,0,0,0,0,0,0,1,2,1,8,7,11,9,11,13,14,14,14,12
4,0,0,0,0,0,0,1,6,7,9,10,13,12,11,13,14,14,15,15,15
5,0,0,0,0,0,1,2,3,5,6,8,9,9,9,9,9,11,11,11,11
6,0,0,0,2,1,1,1,1,1,1,1,1,1,2,2,2,2,2,1,0
7,0,0,0,1,3,2,2,2,1,2,2,3,3,3,3,3,4,4,5,5
8,0,1,0,0,0,0,0,1,1,1,3,4,3,3,4,4,5,6,7,7
9,0,1,0,0,0,0,1,0,0,2,1,1,1,2,4,5,4,4,6,4


harmonic mean ↑
fluency ignored


,0,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19
0,0.000020,0.087500,0.000020,0.066667,0.000010,0.000020,0.000010,0.000010,0.000020,0.000020,0.000020,0.000010,0.000020,0.000020,0.066667,0.066667,0.120000,0.075000,0.080000,0.080000
1,0.000020,0.000020,0.000020,0.000020,0.080000,0.050000,0.066667,0.342857,0.323077,0.142857,0.393750,0.523810,0.634615,0.900000,0.900000,0.924324,0.974359,1.000000,1.000000,0.924324
2,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.100000,0.075000,0.080000,0.000020,0.000020,0.088889,0.169231,0.091667,0.091667,0.090000,0.090000,0.090000,0.090909
3,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.075000,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020
4,0.000020,0.000020,0.000020,0.000020,0.000020,0.000020,0.075000,0.300000,0.254545,0.276923,0.285714,0.000020,0.171429,0.235714,0.173333,0.093333,0.000020,0.000020,0.000020,0.000020
5,0.000020,0.000020,0.000020,0.000020,0.000020,0.075000,0.150000,0.187500,0.291667,0.323077,0.218182,0.276923,0.276923,0.321429,0.321429,0.163636,0.091667,0.000020,0.235714,0.235714
6,0.000020,0.000020,0.000020,0.150000,0.085714,0.085714,0.083333,0.085714,0.085714,0.085714,0.085714,0.083333,0.083333,0.100000,0.066667,0.100000,0.133333,0.133333,0.075000,0.000020
7,0.000020,0.000020,0.000020,0.087500,0.210000,0.150000,0.150000,0.142857,0.083333,0.150000,0.142857,0.200000,0.200000,0.210000,0.210000,0.187500,0.222222,0.266667,0.291667,0.307692
8,0.000020,0.080000,0.000020,0.000020,0.000020,0.000020,0.000020,0.083333,0.085714,0.085714,0.200000,0.254545,0.200000,0.200000,0.254545,0.200000,0.250000,0.240000,0.323077,0.291667
9,0.000020,0.075000,0.000020,0.000020,0.000020,0.000020,0.080000,0.000020,0.000020,0.120000,0.075000,0.066667,0.066667,0.120000,0.222222,0.250000,0.240000,0.240000,0.323077,0.254545
